In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import subprocess
import os
from Bio import SeqIO
from io import StringIO
import tempfile
import numpy as np

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 库导入成功！")

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# BLAST数据库路径（根据您的实际路径修改）
BLAST_DB = r"D:\model_AGORA_add_reactions\my_bacterial_db_all"

# BLAST可执行文件路径（如果已在PATH中，直接写命令名即可）
BLASTN_CMD = "blastn"  # 或完整路径，如 r"C:\Program Files\NCBI\blast-2.16.0+\bin\blastn.exe"

# 检查数据库是否存在
if os.path.exists(BLAST_DB + ".nsq"):
    print(f"✅ BLAST数据库找到: {BLAST_DB}")
else:
    print(f"❌ 数据库不存在: {BLAST_DB}")
    print("请检查路径是否正确")

In [ ]:
def read_genome_fasta(fasta_file):
    """
    读取基因组FASTA文件，返回序列列表
    支持多染色体/contig的基因组文件
    """
    sequences = []
    seq_ids = []
    
    for record in SeqIO.parse(fasta_file, "fasta"):
        sequences.append(str(record.seq))
        seq_ids.append(record.id)
    
    print(f"✅ 读取基因组文件成功！")
    print(f"   - 共 {len(sequences)} 条序列（染色体/contig）")
    print(f"   - 总碱基数: {sum(len(s) for s in sequences):,}")
    print(f"   - 序列ID: {', '.join(seq_ids[:3])}{'...' if len(seq_ids) > 3 else ''}")
    
    return sequences, seq_ids

# 示例：读取基因组文件（请替换为您的实际文件路径）
genome_file = r"D:\model_AGORA_add_reactions\AGORA-Genomes\Abiotrophia_defectiva_ATCC_49176.fasta"
sequences, seq_ids = read_genome_fasta(genome_file)

In [ ]:
def blast_single_sequence(query_seq, seq_id="query", db_path=BLAST_DB, 
                          blastn_cmd=BLASTN_CMD, max_targets=10):
    """
    对单条序列进行BLAST比对
    """
    # 创建临时FASTA文件
    with tempfile.NamedTemporaryFile(mode='w', suffix='.fasta', delete=False) as f:
        f.write(f">{seq_id}\n")
        # 每行80个字符
        for i in range(0, len(query_seq), 80):
            f.write(query_seq[i:i+80] + '\n')
        temp_fasta = f.name
    
    try:
        # 构建BLAST命令
        cmd = [
            blastn_cmd,
            '-db', db_path,
            '-query', temp_fasta,
            '-outfmt', '6',  # 表格格式
            '-max_target_seqs', str(max_targets),
            '-evalue', '1e-5'  # E值阈值
        ]
        
        # 执行命令
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        
        # 解析结果
        if result.stdout.strip():
            df = pd.read_csv(StringIO(result.stdout), sep='\t', header=None,
                            names=['qseqid', 'sseqid', 'pident', 'length', 
                                   'mismatch', 'gapopen', 'qstart', 'qend', 
                                   'sstart', 'send', 'evalue', 'bitscore'])
            return df
        else:
            return pd.DataFrame()  # 无匹配
            
    except subprocess.CalledProcessError as e:
        print(f"❌ BLAST执行失败: {e}")
        print(f"错误信息: {e.stderr}")
        return None
    finally:
        # 清理临时文件
        if os.path.exists(temp_fasta):
            os.remove(temp_fasta)

# 测试单条序列
test_seq = "ATGCCGTTGTTAGATAGCTTCACAGTCGATCATACCCGGATGGAAGCGCCTGCAGTTCGGGTGGCGAAAACAATGAACACCCCGCATGGCGACGCAATCACCGTGTTCGATCTGCGCTTCTGCGTGCCGAACAAAGAAGTGATGCCAGAAAGAGGGATCCATACCCTGGAGCACCTGTTTGCTGGTTTTATGCGTAACCATCTTAACGGTAATGGTGTAGAGATTATCGATATCTCGCCAATGGGCTGCCGCACCGGTTTTTATATGAGTCTGATTGGTACGCCAGATGAGCAGCGTGTTGCTGATGCCTGGAAAGCGGCAATGGAAGACGTGCTGAAAGTGCAGGATCAGAATCAGATCCCGGAACTGAACGTCTACCAGTGTGGCACTTACCAGATGCACTCGTTGCAGGAAGCGCAGGATATTGCGCGTAGCATTCTGGAACGTGACGTACGCATCAACAGCAACGAAGAACTGGCACTGCCGAAAGAGAAGTTGCAGGAACTGCACATCTAG"

print("🔬 测试单条序列BLAST...")
result_df = blast_single_sequence(test_seq, "test_gene")
if result_df is not None and len(result_df) > 0:
    print(f"✅ 找到 {len(result_df)} 条匹配")
    display(result_df.head())
else:
    print("⚠️ 未找到匹配")

In [ ]:
def blast_genome_batch(sequences, seq_ids, db_path=BLAST_DB, 
                       blastn_cmd=BLASTN_CMD, max_targets=10, 
                       chunk_size=100):
    """
    批量比对基因组的所有contig/染色体
    支持大基因组分批处理
    """
    all_results = []
    
    total = len(sequences)
    print(f"🚀 开始批量比对 {total} 条序列...")
    
    for i, (seq, seq_id) in enumerate(zip(sequences, seq_ids)):
        if (i + 1) % max(1, total // 10) == 0 or i == 0:
            print(f"  进度: {i+1}/{total} ({100*(i+1)/total:.1f}%)")
        
        result = blast_single_sequence(seq, seq_id, db_path, blastn_cmd, max_targets)
        
        if result is not None and len(result) > 0:
            all_results.append(result)
    
    if all_results:
        combined = pd.concat(all_results, ignore_index=True)
        print(f"✅ 批量比对完成！共 {len(combined)} 条匹配结果")
        return combined
    else:
        print("⚠️ 未找到任何匹配")
        return pd.DataFrame()

# 模拟基因组数据（实际使用时替换为您的基因组序列）
# 示例：如果您有基因组文件
genome_sequences, genome_ids = read_genome_fasta("D:\model_AGORA_add_reactions\AGORA-Genomes\Abiotrophia_defectiva_ATCC_49176.fasta")
results = blast_genome_batch(genome_sequences, genome_ids)

# 如果没有基因组文件，可以用测试数据演示
# demo_sequences = [test_seq[:200], test_seq[100:300], test_seq[200:400]]
# demo_ids = ["contig_1", "contig_2", "contig_3"]
# results = blast_genome_batch(demo_sequences, demo_ids)

In [ ]:
def parse_blast_results(df):
    """
    解析BLAST结果，提取有用信息
    """
    if df.empty:
        print("⚠️ 没有数据可解析")
        return df
    
    # 提取CDS和物种信息
    df['CDS'] = df['sseqid'].str.split('|').str[0]
    df['species'] = df['sseqid'].str.split('|').str[1]
    df['row'] = df['sseqid'].str.split('|').str[2]
    
    # 添加匹配等级
    def get_match_level(pident):
        if pident >= 99:
            return '完美匹配 (≥99%)'
        elif pident >= 95:
            return '高度相似 (95-99%)'
        elif pident >= 90:
            return '中度相似 (90-95%)'
        else:
            return '低度相似 (<90%)'
    
    df['match_level'] = df['pident'].apply(get_match_level)
    
    # 按E值排序
    df = df.sort_values('evalue')
    
    return df

# 解析结果
if 'results' in locals() and not results.empty:
    parsed_results = parse_blast_results(results)
    print("✅ 结果解析完成")
    display(parsed_results.head())

In [ ]:
def plot_blast_summary(df, title="BLAST比对结果摘要"):
    """
    绘制BLAST结果摘要图
    """
    if df.empty:
        print("⚠️ 没有数据可绘图")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # 1. 物种分布
    species_counts = df['species'].value_counts().head(10)
    axes[0, 0].barh(species_counts.index, species_counts.values, color='steelblue')
    axes[0, 0].set_xlabel('匹配数量')
    axes[0, 0].set_title('Top 10 物种分布')
    
    # 2. 一致性分布
    axes[0, 1].hist(df['pident'], bins=20, color='coral', edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel('一致性 (%)')
    axes[0, 1].set_ylabel('频数')
    axes[0, 1].set_title('一致性分布')
    axes[0, 1].axvline(df['pident'].mean(), color='red', linestyle='--', 
                       label=f'平均: {df["pident"].mean():.1f}%')
    axes[0, 1].legend()
    
    # 3. 匹配等级分布
    level_counts = df['match_level'].value_counts()
    axes[1, 0].pie(level_counts.values, labels=level_counts.index, autopct='%1.1f%%',
                   colors=['#2ecc71', '#3498db', '#f39c12', '#e74c3c'])
    axes[1, 0].set_title('匹配质量分布')
    
    # 4. E值分布 (-log10)
    axes[1, 1].hist(-np.log10(df['evalue']), bins=20, color='purple', edgecolor='black', alpha=0.7)
    axes[1, 1].set_xlabel('-log10(E值)')
    axes[1, 1].set_ylabel('频数')
    axes[1, 1].set_title('E值显著性分布 (越高越显著)')
    
    plt.tight_layout()
    plt.show()

# 绘图
if 'parsed_results' in locals() and not parsed_results.empty:
    plot_blast_summary(parsed_results)